In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# Getting Started with AWS Batch for SageMaker Training jobs

---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook.

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

---

This sample notebook will demonstrate how to submit some simple 'hello world' jobs to an [AWS Batch job queue](https://aws.amazon.com/batch/) using a [ModelTrainer](https://sagemaker.readthedocs.io/en/stable/api/training/model_trainer.html). You can run any of the cells in this notebook interactively to experiment with using your queue. Batch will take care of ensuring your jobs run automatically as your service environment capacity becomes available. 

## Setup and Configure Training Job Variables
We will need a single instance for a short duration for the sample jobs.  Change any of the constant variables below to adjust the example to your liking. 

In [2]:
INSTANCE_TYPE = "ml.g5.xlarge"
INSTANCE_COUNT = 1
MAX_RUN_TIME = 300
TRAINING_JOB_NAME = "hello-world-simple-job"

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logging.getLogger("botocore.client").setLevel(level=logging.WARN)
logger = logging.getLogger(__name__)

from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris

session = Session()

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=session.boto_session.region_name,
    version="2.5",
    instance_type=INSTANCE_TYPE,
    image_scope="training",
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/16/26 22:47:48] INFO     Defaulting to only available Python version: py311                   ]8;id=4071780;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=4071781;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/image_uris.py#615\615]8;;\

## Create Sample Resources
The diagram belows shows the Batch resources we'll create for this example.

![The Resources to Create](batch_getting_started_resources.png "Example Job Queue and Service Environment Resources")

You can use [Batch Console](https://console.aws.amazon.com/batch) to create these resources, or you can run the cell below. The ```create_resources``` function below will skip creating any resources that already exist.

In [4]:
from sagemaker.train.aws_batch.boto_client import get_batch_boto_client
from utils.aws_batch_resource_management import AwsBatchResourceManager, create_resources

# This job queue name needs to match the Job Queue created in AWS Batch.
JOB_QUEUE_NAME = "my-sm-training-fifo-jq"
SERVICE_ENVIRONMENT_NAME = "my-sm-training-fifo-se"

# Create ServiceEnvironment and JobQueue
resource_manager = AwsBatchResourceManager(get_batch_boto_client())
resources = create_resources(
    resource_manager, JOB_QUEUE_NAME, SERVICE_ENVIRONMENT_NAME, max_capacity=1
)

[07/16/26 22:47:49] INFO     Creating ServiceEnvironment:                      ]8;id=4071788;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py\aws_batch_resource_management.py]8;;\:]8;id=4071789;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py#860\860]8;;\
                             my-sm-training-fifo-se                                                                

                    INFO     Waiting for ServiceEnvironment to transition to   ]8;id=4071795;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py\aws_batch_resource_management.py]8;;\:]8;id=4071796;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py#869\869]8;;\
                             VALID...                                                                              

                    INFO     Creating JobQueue: my-sm-training-fifo-jq         ]8;id=4071802;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py\aws_batch_resource_management.py]8;;\:]8;id=4071803;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py#873\873]8;;\

                    INFO     Waiting for JobQueue to transition to VALID...    ]8;id=4071809;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py\aws_batch_resource_management.py]8;;\:]8;id=4071810;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py#885\885]8;;\

[07/16/26 22:47:54] INFO     Resource creation complete:                       ]8;id=4071816;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py\aws_batch_resource_management.py]8;;\:]8;id=4071817;file:///opt/ml/processing/input/utils/aws_batch_resource_management.py#898\898]8;;\
                             Resources(job_queue=Resource(name='my-sm-training                                     
                             -fifo-jq',                                                                            
                             arn='arn:aws:batch:us-west-2:674622101542:job-que                                     
                             ue/my-sm-training-fifo-jq'),                                                          
                             service_environment=Resource(name='my-sm-training                                     
                             -fifo-se',                                                                            
                             arn='arn:aws:batch:us-west-2:674622101542:service                                     
                             -environment/my-sm-training-fifo-se'),                                                
                             scheduling_policy=None, batch_role=None,                                              
                             sagemaker_execution_role=None, quota_shares=None)                                     

## Create Hello World Model Trainer
Now that our resources are created, we'll construct a simple ModelTrainer.

In [5]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

source_code = SourceCode(command="echo 'Hello World'")

model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=TRAINING_JOB_NAME,
    compute=Compute(instance_type=INSTANCE_TYPE, instance_count=INSTANCE_COUNT),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=MAX_RUN_TIME),
)

[07/16/26 22:47:55] INFO     SageMaker session not provided. Using default Session.                  ]8;id=4071824;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4071825;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=4071832;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4071833;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

[07/16/26 22:47:56] WARNING  Could not verify permissions for role                         ]8;id=4071839;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=4071840;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=4071846;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4071847;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=4071853;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4071854;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-674622101542/hello-world-simp                
                             le-job' kms_key_id=None compression_type='GZIP'                                       

                    INFO     Training image URI:                                               ]8;id=4071861;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=4071862;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.5                     
                             -gpu-py311                                                                            

## Create TrainingQueue object
Using our queue is as easy as referring to it by name in the TrainingQueue contructor. The TrainingQueue class within the SageMaker Python SDK provides built in support for working with Batch queues.

In [6]:
from sagemaker.train.aws_batch.training_queue import TrainingQueue, TrainingQueuedJob

# Construct the queue object using the SageMaker Python SDK
queue = TrainingQueue(JOB_QUEUE_NAME)
logger.info(f"Using queue: {queue.queue_name}")

                    INFO     Using queue: my-sm-training-fifo-jq                                    ]8;id=4071869;file:///home/model-server/tmp/ipykernel_431/3455603507.py\3455603507.py]8;;\:]8;id=4071870;file:///home/model-server/tmp/ipykernel_431/3455603507.py#5\5]8;;\

## Submit Some Training Jobs
Submitting your job to the queue is done by calling queue.submit.  This particular job doesn't require any data, but in general, data should be provided by specifying inputs.

In [7]:
# Submit first job
training_queued_job_1: TrainingQueuedJob = queue.submit(training_job=model_trainer, inputs=None)
logger.info(
    f"Submitted job '{training_queued_job_1.job_name}' to TrainingQueue '{queue.queue_name}'"
)

# Submit second job
training_queued_job_2: TrainingQueuedJob = queue.submit(training_job=model_trainer, inputs=None)
logger.info(
    f"Submitted job '{training_queued_job_2.job_name}' to TrainingQueue '{queue.queue_name}'"
)

[07/16/26 22:47:58] INFO     Submitted job 'hello-world-simple-job-20260716224757' to TrainingQueue ]8;id=4071877;file:///home/model-server/tmp/ipykernel_431/2363117991.py\2363117991.py]8;;\:]8;id=4071878;file:///home/model-server/tmp/ipykernel_431/2363117991.py#3\3]8;;\
                             'my-sm-training-fifo-jq'                                                              

[07/16/26 22:47:59] INFO     Submitted job 'hello-world-simple-job-20260716224758' to TrainingQueue ]8;id=4071884;file:///home/model-server/tmp/ipykernel_431/2363117991.py\2363117991.py]8;;\:]8;id=4071885;file:///home/model-server/tmp/ipykernel_431/2363117991.py#9\9]8;;\
                             'my-sm-training-fifo-jq'                                                              

## Terminate a Job in the Queue
This next cell shows how to terminate an in queue job.

In [8]:
logger.info(f"Terminating job: {training_queued_job_2.job_name}")
training_queued_job_2.terminate()

                    INFO     Terminating job: hello-world-simple-job-20260716224758                  ]8;id=4071892;file:///home/model-server/tmp/ipykernel_431/920679439.py\920679439.py]8;;\:]8;id=4071893;file:///home/model-server/tmp/ipykernel_431/920679439.py#1\1]8;;\

## Monitor Job Status
This next cell shows how to list the jobs that have been submitted to the TrainingQueue.  The TrainingQueue can list jobs by status, and each job can be described individually for more details.  Once a TrainingQueuedJob has reached the STARTING status, the logs can be printed from underlying SageMaker training job.

In [9]:
import time
from utils.log_helpers import logs_for_job


def list_jobs_in_training_queue(training_queue: TrainingQueue):
    """
    Lists all jobs in a TrainingQueue grouped by their status.

    This function retrieves jobs with different statuses (SUBMITTED, PENDING, RUNNABLE,
    SCHEDULED, STARTING, RUNNING, SUCCEEDED, FAILED) from the specified TrainingQueue
    and logs their names and current status.

    Args:
        training_queue (TrainingQueue): The TrainingQueue to query for jobs.

    Returns:
        None: This function doesn't return a value but logs job information.
    """
    submitted_jobs = training_queue.list_jobs(status="SUBMITTED")
    pending_jobs = training_queue.list_jobs(status="PENDING")
    runnable_jobs = training_queue.list_jobs(status="RUNNABLE")
    scheduled_jobs = training_queue.list_jobs(status="SCHEDULED")
    starting_jobs = training_queue.list_jobs(status="STARTING")
    running_jobs = training_queue.list_jobs(status="RUNNING")
    completed_jobs = training_queue.list_jobs(status="SUCCEEDED")
    failed_jobs = training_queue.list_jobs(status="FAILED")

    all_jobs = (
        submitted_jobs
        + pending_jobs
        + runnable_jobs
        + scheduled_jobs
        + starting_jobs
        + running_jobs
        + completed_jobs
        + failed_jobs
    )

    for job in all_jobs:
        job_status = job.describe().get("status", "")
        logger.info(f"Job : {job.job_name} is {job_status}")


def monitor_training_queued_job(job: TrainingQueuedJob):
    """
    Monitors a TrainingQueuedJob until it reaches an active or terminal state.

    This function continuously polls the status of the specified TrainingQueuedJob
    until it transitions to one of the following states: STARTING, RUNNING,
    SUCCEEDED, or FAILED. Once the job reaches one of these states, the function
    retrieves and displays the job's logs.

    Args:
        job (TrainingQueuedJob): The TrainingQueuedJob to monitor.

    Returns:
        None: This function doesn't return a value but displays job logs.
    """
    while True:
        job_status = job.describe().get("status", "")

        if job_status in {"STARTING", "RUNNING", "SUCCEEDED", "FAILED"}:
            break

        logger.info(f"Job : {job.job_name} is {job_status}")
        time.sleep(5)

    # Print training job logs
    model_trainer = job.get_model_trainer()
    logs_for_job(model_trainer, wait=True)


logger.info(f"Listing all jobs in queue '{queue.queue_name}'...")
list_jobs_in_training_queue(queue)

logger.info(f"Polling job status for '{training_queued_job_1.job_name}'")
monitor_training_queued_job(training_queued_job_1)

[07/16/26 22:48:00] INFO     Listing all jobs in queue 'my-sm-training-fifo-jq'...                 ]8;id=4071900;file:///home/model-server/tmp/ipykernel_431/2493618177.py\2493618177.py]8;;\:]8;id=4071901;file:///home/model-server/tmp/ipykernel_431/2493618177.py#73\73]8;;\

[07/16/26 22:48:01] INFO     Job : hello-world-simple-job-20260716224757 is SUBMITTED              ]8;id=4071907;file:///home/model-server/tmp/ipykernel_431/2493618177.py\2493618177.py]8;;\:]8;id=4071908;file:///home/model-server/tmp/ipykernel_431/2493618177.py#41\41]8;;\

                    INFO     Job : hello-world-simple-job-20260716224758 is SUBMITTED              ]8;id=4071913;file:///home/model-server/tmp/ipykernel_431/2493618177.py\2493618177.py]8;;\:]8;id=4071914;file:///home/model-server/tmp/ipykernel_431/2493618177.py#41\41]8;;\

                    INFO     Job : hello-world-simple-job-20260715052321 is SUCCEEDED              ]8;id=4071919;file:///home/model-server/tmp/ipykernel_431/2493618177.py\2493618177.py]8;;\:]8;id=4071920;file:///home/model-server/tmp/ipykernel_431/2493618177.py#41\41]8;;\

                    INFO     Job : hello-world-simple-job-20260715054817 is SUCCEEDED              ]8;id=4071925;file:///home/model-server/tmp/ipykernel_431/2493618177.py\2493618177.py]8;;\:]8;id=4071926;file:///home/model-server/tmp/ipykernel_431/2493618177.py#41\41]8;;\

...


2026-07-16 22:55:20 Training - Training image download completed. Training in progress..

.

Starting training script
++ /opt/conda/bin/python3 --version
Python 3.11.10
++ echo /opt/ml/input/config/resourceconfig.json:
++ cat /opt/ml/input/config/resourceconfig.json
/opt/ml/input/config/resourceconfig.json:
++ echo
++ echo /opt/ml/input/config/inputdataconfig.json:
++ cat /opt/ml/input/config/inputdataconfig.json
{"current_host":"algo-1","current_instance_type":"ml.g5.xlarge","current_group_name":"homogeneousCluster","hosts":["algo-1"],"instance_groups":[{"instance_group_name":"homogeneousCluster","instance_type":"ml.g5.xlarge","hosts":["algo-1"]}],"network_interface_name":"eth0","topology":null}
/opt/ml/input/config/inputdataconfig.json:
++ echo
++ echo 'Setting up environment variables'
++ /opt/conda/bin/python3 /opt/ml/input/data/sm_drivers/scripts/environment.py
{"sm_drivers":{"TrainingInputMode":"File","S3DistributionType":"FullyReplicated","RecordWrapperType":"None"}}
Setting up environment variables
No Neurons detected (normal if no neurons installed)
Environment Variab


2026-07-16 22:55:48 Uploading - Uploading generated training model
2026-07-16 22:55:48 Completed - Training job completed


Training seconds: 301
Billable seconds: 301


# Optional: Delete AWS Batch Resources
This shows how to delete the AWS Batch ServiceEnvironment and JobQueue.  This step is completely optional, uncomment the code below to delete the resources created a few steps above.

In [10]:
from utils.aws_batch_resource_management import delete_resources

# delete_resources(resource_manager, resources)

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.


![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-training-queues|sm-training-queues_getting_started_with_model_trainer.ipynb)
